# 01 - Bronze JDBC Ingestion

Loads the three master tables from Azure SQL into the existing Unity Catalog Bronze schema.

- Catalog: `capstone6_telecom`
- Schema: `bronze`
- SQL credentials are read from secret scope `azure-sql-scope` (keys `sql-user`, `sql-pwd`).
- Existing Bronze target tables are appended to, not overwritten.

**Before running:** enter your Azure SQL connection details below. Do not share a notebook containing real passwords.

In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
schema = "bronze"

jdbc_url = "jdbc:sqlserver://capstone6dbserver.database.windows.net:1433;database=capstone6db;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
jdbc_user = dbutils.secrets.get("azure-sql-scope", "sql-user")
jdbc_password = dbutils.secrets.get("azure-sql-scope", "sql-pwd")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
# 2. Azure SQL source tables and Bronze target tables
tables = {
    "dbo.tower_master": "bronze_tower_master",
    "dbo.network_topology_master": "bronze_network_topology_master",
    "dbo.region_lookup": "bronze_region_lookup"
}

In [0]:
# 3. Read each SQL table and append it to Bronze
from pyspark.sql import functions as F

for source_table, target_table in tables.items():

    source_df = (spark.read
        .format("jdbc")
        .option("url", jdbc_url).option("encrypt", "true").option("trustServerCertificate", "true")
        .option("dbtable", source_table)
        .option("user", jdbc_user)
        .option("password", jdbc_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())

    bronze_df = (source_df
        .withColumn("_source_system", F.lit("Azure SQL"))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_ingested_at", F.current_timestamp()))

    (bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{schema}.{target_table}"))

    print(f"Loaded {source_table} into {target_table}")

In [0]:
# 4. Verify the three Bronze tables
for target_table in tables.values():
    print(f"Table: {target_table}")
    display(spark.table(f"{catalog}.{schema}.{target_table}").limit(5))